# 🎬 AI Movie Translate (v2.2) — Google Colab GPU Edition
### ⚡ 100% Full Movie Dialogue Translation & Dubbing Pipeline
**Autonomous Multi-Agent • Full Dialogue Translation • Colloquial Spoken Burmese • Free Cloud GPU**

[![GitHub](https://img.shields.io/badge/GitHub-paipai1999%2Fai--translate--agent-blue?logo=github)](https://github.com/paipai1999/ai-translate-agent)
[![Colab](https://img.shields.io/badge/Google%20Colab-T4%20GPU%20Ready-orange?logo=googlecolab)](https://colab.research.google.com)
[![License](https://img.shields.io/badge/License-MIT-green)](https://github.com/paipai1999/ai-translate-agent/blob/main/LICENSE)

---

### 📌 အရေးကြီးသော ကြိုတင်ပြင်ဆင်မှု (Pre-requisite):
ဤစနစ်သည် **NVIDIA GPU** ဖြင့် Run လျှင် အသံဖမ်းယူမှု (Whisper STT)၊ Demucs အသံခွဲထုတ်မှု နှင့် Video Rendering များ **၅ ဆ မှ ၁၀ ဆ ခန့် ပိုမိုမြန်ဆန်** ပါသည်။
1. မီးနူးဘားမှ: **Runtime** > **Change runtime type** ကို နှိပ်ပါ။
2. **Hardware accelerator** တွင် **`T4 GPU`** (အခမဲ့) ကို ရွေးပေးပါ။
3. **Save** ကို နှိပ်ပါ။

> 💡 **အကြံပြုချက်:** အောက်တွင် **နည်းလမ်း (၂) မျိုး** ဖြင့် အသုံးပြုနိုင်ပါသည်:
> * **နည်းလမ်း (၁):** အောက်ပါ **⚡ 1-Click All-in-One** ကွက်တွင် Link ထည့်ပြီး Play (▶️) တစ်ချက်နှိပ်ရုံဖြင့် အပြီးသတ် ဗီဒီယို ထွက်လာစေခြင်း။
> * **နည်းလမ်း (၂):** **Step 4 (Option B)** သို့ သွားပြီး လှပသော **Web UI Dashboard** ဖွင့်ကာ Browser ပေါ်မှ အသုံးပြုခြင်း။


## ⚡ 1-Click All-in-One Translation (အမြန်ဆုံး နည်းလမ်း)
အောက်ပါ Form တွင် Video Link နှင့် Gemini API Key ထည့်ပြီး Play (▶️) နှိပ်လိုက်ရုံဖြင့် အလိုအလျောက် စတင်ပါမည်:

In [ ]:
# @title ⚡ 1-Click Full Movie Translation Runner
VIDEO_URL = "https://www.youtube.com/watch?v=5VRSIZwxJso" # @param {type:"string"}
GEMINI_API_KEY = "" # @param {type:"string"}
VOICE_ENGINE = "edge_tts" # @param ["edge_tts", "f5_tts"]
CUSTOM_THUMBNAIL_TITLE = "" # @param {type:"string"}
WATERMARK_ENABLED = True # @param {type:"boolean"}
WATERMARK_TEXT = "PAI AI Movie Translate" # @param {type:"string"}
WATERMARK_OPACITY = 0.4 # @param {type:"number"}

import os, sys, json, subprocess

# 1. Setup repository & ensure absolute working directory
project_dir = "/content/ai-translate-agent"
if not os.path.exists(project_dir):
    print("[*] 1/4 Cloning repository...")
    !git clone https://github.com/paipai1999/ai-translate-agent.git /content/ai-translate-agent
else:
    print("[*] 1/4 Updating repository to latest code...")
    !cd /content/ai-translate-agent && git reset --hard HEAD && git pull origin main

os.chdir(project_dir)
%cd /content/ai-translate-agent

if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

os.makedirs(os.path.join(project_dir, "temp"), exist_ok=True)
os.makedirs(os.path.join(project_dir, "outputs"), exist_ok=True)
os.makedirs(os.path.join(project_dir, "movies"), exist_ok=True)

# 2. Install dependencies if not already done
if not os.path.exists("/usr/share/fonts/truetype/padauk/Padauk.ttf"):
    print("[*] 2/4 Installing FFmpeg & Myanmar Padauk fonts...")
    !apt-get update -qq && apt-get install -y -qq ffmpeg fonts-sil-padauk fonts-noto-cjk fonts-noto-core > /dev/null 2>&1
    !fc-cache -f > /dev/null 2>&1
    print("[*] Installing Python packages (this takes ~1-2 minutes)...")
    !pip install -q -r requirements.txt

# 3. Save config using absolute path & fallback
print("[*] 3/4 Configuring pipeline...")
config_path = os.path.join(project_dir, "config.json")
if os.path.exists(config_path):
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)
else:
    cfg = {}

cfg.setdefault("pipeline", {})["language"] = "burmese"
cfg.setdefault("voice", {})["engine"] = VOICE_ENGINE
if GEMINI_API_KEY.strip():
    cfg.setdefault("gemini", {})["api_keys"] = [GEMINI_API_KEY.strip()]
cfg.setdefault("watermark", {})["enabled"] = WATERMARK_ENABLED
cfg["watermark"]["text"] = WATERMARK_TEXT
cfg["watermark"]["opacity"] = WATERMARK_OPACITY
cfg.setdefault("subtitle_overlay", {})["font_name"] = "Padauk"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=4)

# 4. Run pipeline reliably via subprocess inside project_dir
print(f"[*] 4/4 Starting autonomous dialogue translation for: {VIDEO_URL}")
cmd = [sys.executable, "main.py", VIDEO_URL.strip(), "-l", "burmese", "-e", VOICE_ENGINE]
if CUSTOM_THUMBNAIL_TITLE.strip():
    cmd.extend(["--thumb-title", CUSTOM_THUMBNAIL_TITLE.strip()])
if not WATERMARK_ENABLED:
    cmd.append("--no-watermark")
elif WATERMARK_TEXT.strip():
    cmd.extend(["--watermark-text", WATERMARK_TEXT.strip()])

subprocess.run(cmd, cwd=project_dir, check=True)


---
## 🔍 Pre-Flight: Check GPU & Hardware Acceleration
T4 GPU ချိတ်ဆက်ထားခြင်း ရှိမရှိ စစ်ဆေးပါ:

In [ ]:
import torch

print("=== 🖥️ HARDWARE SPECIFICATIONS ===")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ Active GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    print(f"✅ PyTorch CUDA Version: {torch.version.cuda}")
    !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
else:
    print("⚠️ WARNING: No GPU detected! Currently running on CPU.")
    print("👉 Please go to: Runtime > Change runtime type > Select 'T4 GPU' for maximum speed.")


## 📦 Step 1: Clone Repository & Setup Workspace
Google Drive ချိတ်ဆက်စရာမလိုဘဲ Colab ၏ High-Speed Local SSD ပေါ်တွင် တိုက်ရိုက် Clone လုပ်ပြီး Run ပါမည်။


In [ ]:
import os, sys

project_dir = "/content/ai-translate-agent"
if not os.path.exists(project_dir):
    print("[*] Cloning repository...")
    !git clone https://github.com/paipai1999/ai-translate-agent.git /content/ai-translate-agent
else:
    print("[*] Existing repository found, fetching latest updates...")
    !cd /content/ai-translate-agent && git reset --hard HEAD && git pull origin main

# Explicitly set Python working directory
os.chdir(project_dir)
%cd /content/ai-translate-agent

if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

os.makedirs("temp", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("movies", exist_ok=True)
print("\n✅ Working Directory: " + os.getcwd())
print("⚡ Running purely on Colab High-Speed Local SSD (Zero Google Drive needed)")


## 🛠️ Step 2: Install System Dependencies & Myanmar Fonts
FFmpeg၊ Myanmar Fonts (Padauk/Noto) နှင့် Python Libraries များကို တင်သွင်းပါမည်။


In [ ]:
print("[*] Installing Linux packages (FFmpeg, Myanmar Padauk Fonts)...")
!apt-get update -qq
!apt-get install -y -qq ffmpeg fonts-sil-padauk fonts-noto-cjk fonts-noto-core > /dev/null 2>&1
!fc-cache -f > /dev/null 2>&1

print("[*] Installing Python libraries from requirements.txt...")
!pip install -q -r requirements.txt

print("\n✅ System & Python dependencies successfully installed!")


## ⚙️ Step 3: Interactive Configuration Form
အောက်ပါ Form တွင် သင့် Gemini API Key နှင့် Settings များကို ရွေးချယ်ပြီး Run ပေးပါ:


In [ ]:
# @title 🎬 Video & AI Pipeline Settings { run: "auto" }

# @markdown ### 🔑 1. Google Gemini API Key
# @markdown အခမဲ့ API Key ကို [Google AI Studio](https://aistudio.google.com/app/apikey) တွင် ရယူပါ:
GEMINI_API_KEY = "" # @param {type:"string"}

# @markdown ### 🗣️ 2. Language & Voiceover Engine
LANGUAGE = "burmese" # @param ["burmese", "english"]
VOICE_ENGINE = "edge_tts" # @param ["edge_tts", "f5_tts"]
MYANMAR_VOICE = "my-MM-ThihaNeural" # @param ["my-MM-ThihaNeural", "my-MM-NilarNeural"]
ENGLISH_VOICE = "en-US-GuyNeural" # @param ["en-US-GuyNeural", "en-US-JennyNeural"]

# @markdown ### 🛡️ 3. Branding & Watermark Settings
WATERMARK_ENABLED = True # @param {type:"boolean"}
WATERMARK_TEXT = "PAI AI Movie Translate" # @param {type:"string"}
WATERMARK_OPACITY = 0.4 # @param {type:"number"}

# @markdown ### 🤖 4. Model & Processing Settings
WHISPER_MODEL = "small" # @param ["base", "small", "medium", "large-v3"]
GEMINI_MODEL = "gemini-3.5-flash-lite" # @param ["gemini-3.5-flash-lite", "gemini-3.6-flash"]
ANTI_COPYRIGHT = True # @param {type:"boolean"}
SUBTITLE_BLUR = True # @param {type:"boolean"}

import json, os, torch

project_dir = "/content/ai-translate-agent" if os.path.exists("/content/ai-translate-agent") else os.getcwd()
os.chdir(project_dir)
config_path = os.path.join(project_dir, "config.json")

if not GEMINI_API_KEY.strip():
    print("⚠️ သတိပေးချက်: GEMINI_API_KEY မထည့်ရသေးပါ။ https://aistudio.google.com/app/apikey မှ Key ထည့်ပေးပါ။")

if os.path.exists(config_path):
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)
else:
    cfg = {}

if GEMINI_API_KEY.strip():
    cfg.setdefault("gemini", {})["api_keys"] = [GEMINI_API_KEY.strip()]
cfg.setdefault("gemini", {})["model"] = GEMINI_MODEL
cfg.setdefault("pipeline", {})["language"] = LANGUAGE
cfg["pipeline"]["whisper_model"] = WHISPER_MODEL
cfg.setdefault("voice", {})["engine"] = VOICE_ENGINE
cfg["voice"]["tts_voice_mm"] = MYANMAR_VOICE
cfg["voice"]["tts_voice_en"] = ENGLISH_VOICE
cfg["voice"]["tts_voice"] = MYANMAR_VOICE if LANGUAGE == "burmese" else ENGLISH_VOICE
cfg.setdefault("copyright_protection", {})["enabled"] = ANTI_COPYRIGHT
cfg.setdefault("subtitle_blur", {})["enabled"] = SUBTITLE_BLUR
cfg.setdefault("watermark", {})["enabled"] = WATERMARK_ENABLED
cfg["watermark"]["text"] = WATERMARK_TEXT
cfg["watermark"]["opacity"] = WATERMARK_OPACITY
cfg.setdefault("subtitle_overlay", {})["font_name"] = "Padauk"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=4)

print("\n✅ config.json updated successfully!")
print(f"🎯 Language: {LANGUAGE.upper()} | Voice Engine: {VOICE_ENGINE.upper()} | LLM: {GEMINI_MODEL}")


## 🚀 Step 4 (Option A): 1-Click Video Execution (Command Line)
YouTube Link သို့မဟုတ် Video URL ကို ထည့်သွင်းပြီး တိုက်ရိုက် Run နိုင်ပါသည်:


In [ ]:
# @title 🎬 Run Autonomous Full Movie Dialogue Translation Pipeline
VIDEO_URL = "https://www.youtube.com/watch?v=5VRSIZwxJso" # @param {type:"string"}
CUSTOM_THUMBNAIL_TITLE = "" # @param {type:"string"}

import os, sys, subprocess

project_dir = "/content/ai-translate-agent" if os.path.exists("/content/ai-translate-agent") else os.getcwd()
os.chdir(project_dir)

if not VIDEO_URL.strip():
    print("❌ ကျေးဇူးပြု၍ ဗီဒီယို Link ထည့်ပေးပါ။")
else:
    print(f"[*] Starting autonomous full dialogue translation for: {VIDEO_URL}")
    cmd = [sys.executable, "main.py", VIDEO_URL.strip(), "-l", LANGUAGE, "-e", VOICE_ENGINE]
    if CUSTOM_THUMBNAIL_TITLE.strip():
        cmd.extend(["--thumb-title", CUSTOM_THUMBNAIL_TITLE.strip()])
    if not WATERMARK_ENABLED:
        cmd.append("--no-watermark")
    elif WATERMARK_TEXT.strip():
        cmd.extend(["--watermark-text", WATERMARK_TEXT.strip()])
    subprocess.run(cmd, cwd=project_dir, check=True)


## 🌐 Step 4 (Option B): Launch Visual Web Dashboard (Interactive)
Browser ပေါ်တွင် Visual Interface ဖြင့် အသုံးပြုလိုပါက ဤ Cell ကို Run ပါ:


In [ ]:
# @title 🌐 Launch Web UI with Cloudflare Tunnel (Click to Open)
import os, sys, subprocess, time, re
from IPython.display import display, HTML, Javascript

project_dir = "/content/ai-translate-agent" if os.path.exists("/content/ai-translate-agent") else os.getcwd()
os.chdir(project_dir)

if not os.path.exists("/usr/local/bin/cloudflared"):
    print("[*] Installing Cloudflared Tunnel...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# Stop any existing server processes
!pkill -f "web_ui.py" || true
!pkill -f "cloudflared" || true

print("[*] Starting Web UI Background Server...")
server_proc = subprocess.Popen(
    [sys.executable, "web_ui.py", "--port", "5000"],
    cwd=project_dir,
    stdout=open("/content/web_ui.log", "w"),
    stderr=subprocess.STDOUT
)
time.sleep(4)

if server_proc.poll() is not None:
    print("❌ Web UI failed to start! Crash logs:")
    with open("/content/web_ui.log", "r") as f:
        print(f.read())
else:
    print("[*] Establishing Cloudflare Tunnel...")
    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    
    tunnel_url = None
    start_t = time.time()
    while time.time() - start_t < 30:
        line = tunnel_proc.stdout.readline()
        if not line and tunnel_proc.poll() is not None:
            break
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
            
    if tunnel_url:
        display(HTML(f"""
        <div style="background-color: #161b22; border: 2px solid #58a6ff; border-radius: 12px; padding: 24px; text-align: center; margin: 20px 0; font-family: sans-serif;">
            <h2 style="color: #58a6ff; margin-bottom: 8px;">🚀 Web UI Dashboard အသင့်ဖြစ်ပါပြီ!</h2>
            <p style="color: #c9d1d9; font-size: 15px; margin-bottom: 16px;">အောက်ပါ ခလုတ်ကြီးကို နှိပ်၍ Browser Tab အသစ်ဖြင့် Web UI ကို တိုက်ရိုက် ဖွင့်ပါ 👇</p>
            <a href="{tunnel_url}" target="_blank" style="background: linear-gradient(135deg, #1f6feb, #238636); color: #ffffff; font-weight: bold; font-size: 18px; padding: 14px 28px; border-radius: 8px; text-decoration: none; display: inline-block; box-shadow: 0 4px 12px rgba(0,0,0,0.4);">
                🌐 Open Web UI Dashboard ↗️
            </a>
            <p style="color: #8b949e; font-size: 12px; margin-top: 14px;">Direct Link: <a href="{tunnel_url}" target="_blank" style="color: #58a6ff;">{tunnel_url}</a></p>
        </div>
        """))
        print(f"\n👉 Direct URL: {tunnel_url}")
        # Auto-pop open browser tab
        try:
            display(Javascript(f'window.open("{tunnel_url}", "_blank");'))
        except Exception:
            pass
        tunnel_proc.wait()
    else:
        print("❌ Could not obtain tunnel URL. Check logs:")
        with open("/content/web_ui.log", "r") as f:
            print(f.read())


## 🎬 Step 5: Output Player & Results Preview
ထွက်ရှိလာသော Dubbed Video၊ Thumbnail၊ Subtitle နှင့် Script များကို Colab ပေါ်တွင် တိုက်ရိုက် ကြည့်ရှု/Download ဆွဲပါ:


In [ ]:
import os
import json
from IPython.display import display, HTML, Video, Image

outputs_dir = "outputs"
if not os.path.exists(outputs_dir) or not os.listdir(outputs_dir):
    print("ℹ️ No completed movies found yet in outputs/.")
else:
    for movie in sorted(os.listdir(outputs_dir)):
        movie_dir = os.path.join(outputs_dir, movie)
        if not os.path.isdir(movie_dir): continue
        
        final_mp4 = os.path.join(movie_dir, "final_recap.mp4")
        thumb_jpg = os.path.join(movie_dir, "thumbnail.jpg")
        state_json = os.path.join(movie_dir, "state.json")
        script_txt = os.path.join(movie_dir, "final_recap_script.txt")
        
        print(f"\n{'='*60}")
        print(f"🎥 MOVIE: {movie}")
        print(f"{'='*60}")
        
        if os.path.exists(state_json):
            with open(state_json, 'r', encoding='utf-8') as f:
                s = json.load(f)
            total_dur = s.get("total_duration_formatted", "N/A")
            print(f"⏱️ Total Duration: {total_dur} | Phase: {s.get('current_phase')} | Progress: {s.get('progress')}%")
            print("⏱️ Timing Breakdown:")
            for phase, dur in s.get("phase_durations", {}).items():
                print(f"   • {phase:<40}: {dur}s")
                
        if os.path.exists(thumb_jpg):
            print(f"\n🖼️ Generated High-CTR Thumbnail:")
            display(Image(thumb_jpg, width=480))
            
        if os.path.exists(final_mp4):
            size_mb = os.path.getsize(final_mp4) / (1024 * 1024)
            print(f"\n✅ Final Dubbed Video Ready ({size_mb:.1f} MB): {final_mp4}")
            display(Video(final_mp4, embed=True, width=640))
            
        if os.path.exists(script_txt):
            print(f"\n📜 Translated Burmese Dialogue Script Preview (First 500 characters):")
            with open(script_txt, 'r', encoding='utf-8') as f:
                print(f.read()[:500] + "...")
